# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and name
print("Available RecordSets:")
recordset_ids = []
for rset in metadata.record_sets:
    print(f"  @id: {rset['@id']}, name: {rset.get('name', '(no name)')}")
    recordset_ids.append(rset['@id'])

# Show fields and columns for first record set as an example
if recordset_ids:
    first_record_set_id = recordset_ids[0]
    rset = [rs for rs in metadata.record_sets if rs['@id'] == first_record_set_id][0]
    print(f"\nFields in RecordSet {first_record_set_id}:")
    for field in rset.get('field', []):
        print(f"  field @id: {field['@id']} | name: {field.get('name', '(no name)')}")

    # For each field, list any columns present
    for field in rset.get('field', []):
        columns = field.get('column', [])
        if columns:
            for col in columns:
                print(f"    column @id: {col['@id']} | name: {col.get('name', '(no name)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in recordset_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Could not load data for RecordSet {record_set_id}: {e}")

if recordset_ids:
    rs_id = recordset_ids[0]
    print(f"\nColumns in DataFrame for RecordSet {rs_id}:")
    if not dataframes[rs_id].empty:
        print(dataframes[rs_id].columns.tolist())
        display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis (replace with relevant @id seen in cell 5 if needed)
rs_id = recordset_ids[0] if recordset_ids else None
df = dataframes.get(rs_id)
if df is not None and not df.empty:
    # Try to select a numeric column automatically
    numeric_columns = df.select_dtypes(include=['number']).columns
    if len(numeric_columns) > 0:
        numeric_field = numeric_columns[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        numeric_field = None
        print("No numeric fields detected.")

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'iufc' else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical/text field (falling back to first object-type column if available)
        group_field = None
        object_cols = df.select_dtypes(include=['object', 'category']).columns
        for col in object_cols:
            if df[col].nunique() > 1 and df[col].nunique() < 30:
                group_field = col
                break
        if group_field:
            print(f"Grouped data by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and not df.empty and numeric_field:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=20)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        filtered_df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
        plt.suptitle("")  # Remove default title
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded dataset metadata and extracted records using the `mlcroissant` library.
- Explored the available record sets and fields via their `@id`s for transparent and modular access.
- Demonstrated filtering and normalization of a numeric field, as well as grouping and visualization (where applicable).
- For advanced analyses, consult the schema fields by `@id` and tailor extraction or processing as needed.

**Note:** For more complex use cases and to ensure all relevant fields are discovered, refer to the online Croissant schema for this dataset and use the `@id` values as references in your code.